Delete Duplicate Schema Evolution DataTypes Handleing / Type casting Null Records Handling Triming Case Conversion Maintain Order of Data

In [0]:
#  Dropdown widget for environment selection
dbutils.widgets.dropdown(name="environment",defaultValue="dev",choices=["dev", "prd", "qa"],label="select Environment"
)
# Get selected environment
env = dbutils.widgets.get("environment")

#Define dynamic table names and source file location
silverTablName = f"saleslake_{env}.silver_{env}.cleanedsales"
bronzeTablName = f"saleslake_{env}.bronze_{env}.rawsales"
srcFileLoc = f"/Volumes/saleslake_{env}/silver_{env}/vol_saleslake_src_files_{env}/daily_sales/"

In [0]:
#Insert into Silver with cleansing rules
spark.sql(f"""
INSERT INTO {silverTablName}
SELECT DISTINCT
CAST(TRIM(sale_id)AS INTEGER)as sale_id,
UPPER(TRIM(product)) as product,
UPPER(TRIM(category))as category,
CAST(TRIM(quantity)AS INTEGER)as quantity,
CAST(TRIM(price)AS DOUBLE)as price,
TO_DATE(TRIM(sale_date),'yyyy-MM-dd')as sale_date,
UPPER(TRIM(region))as region,
CURRENT_TIMESTAMP() as ingest_ts
FROM {bronzeTablName}
WHERE ingest_ts > (
                     SELECT coalesce(MAX(ingest_ts),TO_DATE('1990-01-01','yyyy-MM-dd')) 
                     FROM {silverTablName}
                    )
 ORDER BY CAST(TRIM(sale_id) AS INTEGER);
 """)